## Importando os dados

In [0]:
from pathlib import Path
import pandas as pd
import tarfile
import urllib.request

In [0]:
def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
    with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing = load_housing_data()

In [0]:
housing.head()

Colocando em portugues cada coluna:

(longitude, latitude, idade média do imovel, total de comodos, total de quartos, populacao, familia, renda_media, valor medio do imovel e proximidade do mar)

In [0]:
housing.info()

In [0]:
## descobrindo os valores categoricos de ocean_proximity
housing["ocean_proximity"].value_counts()

In [0]:
## describe básico das colunas
housing.describe()

In [0]:
## plotando um histograma para entender a natureza dos dados
import matplotlib.pyplot as plt

In [0]:
housing.hist(bins=50, figsize=(20,15))
plt.show()

Algumas variaveis conhecidas como tail-heavy, podem dificultar o desempenho de alguns algoritmos, deveremos tratar isso mais tarde

## Criando um conjunto de teste

In [0]:
import numpy as np

In [0]:
def split_train_test(data, test_ratio):
    shuffled_indices = np.random.permutation(len(data)) ## separando meus indices embaralhados 
    test_set_size = int(len(data) * test_ratio) ## define o tamanho do meu set de teste para meu test_ratio
    test_indices = shuffled_indices[:test_set_size] ## define que só vou pegar test_ratio valores do meu array embaralhado
    train_indices = shuffled_indices[test_set_size:] ## pego o resto do array, exceto, o test_indices
    ## data.iloc[train_indices] pega todas as linhas que foram separadas antes, ou seja, 80% do dataset
    ## data.iloc[test_set_size] pega todas as linhas que foram separadas para teste, ou seja, 20%
    return data.iloc[train_indices], data.iloc[test_indices] 


In [0]:
train_set, test_set = split_train_test(housing, 0.2)

In [0]:
len(train_set)

In [0]:
len(test_set)

In [0]:
from zlib import crc32

In [0]:
def test_set_check(identifier, test_ratio):
    # criando os hash e vendo se é menor que o test ratio
    # menor que test ratio = teste
    # maior que test_ratio = treino
    return crc32(np.int64(identifier)) & 0xffffffff < test_ratio * 2**32 

In [0]:
def split_train_test_by_id(data, test_ratio, id_column):
    ids = data[id_column] # transformando meu index em coluna
    in_test_set = ids.apply(lambda id_: test_set_check(id_, test_ratio)) # aplicando a funcao em todos os id
    return data.loc[~in_test_set], data.loc[in_test_set] # os valores boolenaos retornados pela funcao são colocados no df

In [0]:
housing_with_id = housing.reset_index() # adicionando um index

In [0]:
train_set, test_set = split_train_test_by_id(housing_with_id, 0.2, "index")

Utilizando outras caracteristicas para criar o ID's, além dos indices

In [0]:
housing_with_id.head()

In [0]:
housing_with_id["id"] = housing["longitude"] *1000 + housing["latitude"]

In [0]:
train_set, test_set = split_train_test_by_id(housing_with_id, 0.2, "id")

Podemos fazer isso com o sklearn também

Quando o dataset for pequeno, tome cuidado para não introduzir um viés de amostragem  

Amostragem estratificada, manter a proporção da amostra em relação a população 

### Amostragem estratificada

In [0]:
housing["income_cat"] = pd.cut(housing["median_income"],
                               bins=[0.,1.5,3.0,4.5, 6., np.inf],
                               labels=[1,2,3,4,5])
housing["income_cat"].hist()
plt.show()

In [0]:
from sklearn.model_selection import StratifiedShuffleSplit

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42) # criando minha amostra estratificada
# primeiro vamos pegar os indices referentes a teste e treino
# a função vai pegar o nosso dataframe e usar o gabarito(nesse caso o housing["income_cat"]) para fazer a amostra estratificada
for train_index, test_index in split.split(housing, housing["income_cat"]): 
  strat_train_set = housing.loc[train_index] # indice estratificada de treino, usamos o loc e não o iloc, pois o iloc pegaria os valores padrões de indice
  strat_test_set = housing.loc[test_index]
strat_test_set["income_cat"].value_counts() / len(strat_test_set)

Removendo os dados income_cat

In [0]:
for set_ in (strat_test_set, strat_train_set):
  set_.drop("income_cat", axis=1, inplace=True)